# UKAN Training for Massachusetts Buildings Segmentation

Semantic segmentation on the Massachusetts Buildings dataset with paired images and labels.

https://github.com/JaouadT/KANU_Net

In [1]:
import os
import glob
import time
import csv
import random
from pathlib import Path

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torch.nn.functional as F
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
PROJECT_ROOT = os.path.abspath("..")
DATASET_ROOT = os.path.join(PROJECT_ROOT, "dataset", "archive")
DATASET_VARIANT = "tiff"  # "tiff" or "png"
MODEL_SAVE_DIR = os.path.join(PROJECT_ROOT, "KAN-UNET", "models")
os.makedirs(MODEL_SAVE_DIR, exist_ok=True)

IMG_SIZE = 256
NUM_CLASSES = 2
CLASS_NAMES = ["background", "building"]
IGNORE_INDEX = None  # set to 0 to ignore background in metrics

BATCH_SIZE = 1
LEARNING_RATE = 1e-4
EPOCHS = 50
USE_AMP = torch.cuda.is_available()
ACCUM_STEPS = 8
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

TRAIN_IMG_DIR = os.path.join(DATASET_ROOT, DATASET_VARIANT, "train")
TRAIN_MASK_DIR = os.path.join(DATASET_ROOT, DATASET_VARIANT, "train_labels")
VAL_IMG_DIR = os.path.join(DATASET_ROOT, DATASET_VARIANT, "val")
VAL_MASK_DIR = os.path.join(DATASET_ROOT, DATASET_VARIANT, "val_labels")
TEST_IMG_DIR = os.path.join(DATASET_ROOT, DATASET_VARIANT, "test")
TEST_MASK_DIR = os.path.join(DATASET_ROOT, DATASET_VARIANT, "test_labels")

def _count_images(path, exts):
    count = 0
    for ext in exts:
        count += len(glob.glob(os.path.join(path, f"*.{ext}")))
    return count

exts = ["tif", "tiff"] if DATASET_VARIANT == "tiff" else ["png", "jpg", "jpeg"]

for p in [TRAIN_IMG_DIR, TRAIN_MASK_DIR, VAL_IMG_DIR, VAL_MASK_DIR, TEST_IMG_DIR, TEST_MASK_DIR]:
    if not os.path.isdir(p):
        raise FileNotFoundError(f"Missing directory: {p}")

print(f"Device: {DEVICE}")
print("Dataset variant:", DATASET_VARIANT)
print("Train images:", _count_images(TRAIN_IMG_DIR, exts), "Train labels:", _count_images(TRAIN_MASK_DIR, exts))
print("Val images:", _count_images(VAL_IMG_DIR, exts), "Val labels:", _count_images(VAL_MASK_DIR, exts))
print("Test images:", _count_images(TEST_IMG_DIR, exts), "Test labels:", _count_images(TEST_MASK_DIR, exts))

Device: cuda
Dataset variant: tiff
Train images: 137 Train labels: 137
Val images: 4 Val labels: 4
Test images: 10 Test labels: 10


/tmp/ipykernel_16369/4244501707.py:17: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)


## Dataset Loader

In [3]:
class MassachusettsBuildingsDataset(Dataset):
    def __init__(self, image_dir, mask_dir, img_size=256, variant="tiff"):
        self.image_dir = Path(image_dir)
        self.mask_dir = Path(mask_dir)
        self.img_size = img_size
        self.exts = ["tif", "tiff"] if variant == "tiff" else ["png", "jpg", "jpeg"]

        self.pairs = self._collect_pairs()
        if len(self.pairs) == 0:
            raise RuntimeError(f"No image/mask pairs found in {image_dir} and {mask_dir}")

        self.img_transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
        ])
        self.mask_resize = transforms.Resize((img_size, img_size), interpolation=Image.NEAREST)

    def _collect_pairs(self):
        img_paths = []
        for ext in self.exts:
            img_paths.extend(sorted(self.image_dir.glob(f"*.{ext}")))
        mask_map = {}
        for ext in self.exts:
            for p in self.mask_dir.glob(f"*.{ext}"):
                mask_map[p.stem] = p
        pairs = []
        for img_path in img_paths:
            mask_path = mask_map.get(img_path.stem)
            if mask_path is not None:
                pairs.append((img_path, mask_path))
        return pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]
        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path)

        image = self.img_transform(image)
        mask = self.mask_resize(mask)
        mask = np.array(mask)
        if mask.ndim == 3:
            mask = mask[..., 0]
        mask = (mask > 127).astype(np.int64)
        mask = torch.from_numpy(mask)
        return image, mask

## KANU_Net Model (JaouadT/KANU_Net)

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import List, Tuple, Union


class PolynomialFunction(nn.Module):
    def __init__(self, degree: int = 3):
        super().__init__()
        self.degree = degree

    def forward(self, x):
        return torch.stack([x ** i for i in range(self.degree)], dim=-1)


class BSplineFunction(nn.Module):
    def __init__(self, grid_min: float = -2.0, grid_max: float = 2.0, degree: int = 3, num_basis: int = 8):
        super(BSplineFunction, self).__init__()
        self.degree = degree
        self.num_basis = num_basis
        self.knots = torch.linspace(grid_min, grid_max, num_basis + degree + 1)  # Uniform knots

    def basis_function(self, i, k, t):
        if k == 0:
            return ((self.knots[i] <= t) & (t < self.knots[i + 1])).float()
        else:
            left_num = (t - self.knots[i]) * self.basis_function(i, k - 1, t)
            left_den = self.knots[i + k] - self.knots[i]
            left = left_num / left_den if left_den != 0 else 0

            right_num = (self.knots[i + k + 1] - t) * self.basis_function(i + 1, k - 1, t)
            right_den = self.knots[i + k + 1] - self.knots[i + 1]
            right = right_num / right_den if right_den != 0 else 0

            return left + right

    def forward(self, x):
        x = x.squeeze()  # Assuming x is of shape (B, 1)
        basis_functions = torch.stack(
            [self.basis_function(i, self.degree, x) for i in range(self.num_basis)], dim=-1
        )
        return basis_functions


class ChebyshevFunction(nn.Module):
    def __init__(self, degree: int = 4):
        super(ChebyshevFunction, self).__init__()
        self.degree = degree

    def forward(self, x):
        chebyshev_polynomials = [torch.ones_like(x), x]
        for n in range(2, self.degree):
            chebyshev_polynomials.append(2 * x * chebyshev_polynomials[-1] - chebyshev_polynomials[-2])
        return torch.stack(chebyshev_polynomials, dim=-1)


class FourierBasisFunction(nn.Module):
    def __init__(self, num_frequencies: int = 4, period: float = 1.0):
        super(FourierBasisFunction, self).__init__()
        assert num_frequencies % 2 == 0, "num_frequencies must be even"
        self.num_frequencies = num_frequencies
        self.period = nn.Parameter(torch.Tensor([period]), requires_grad=False)

    def forward(self, x):
        frequencies = torch.arange(1, self.num_frequencies // 2 + 1, device=x.device)
        sin_components = torch.sin(2 * torch.pi * frequencies * x[..., None] / self.period)
        cos_components = torch.cos(2 * torch.pi * frequencies * x[..., None] / self.period)
        basis_functions = torch.cat([sin_components, cos_components], dim=-1)
        return basis_functions


class RadialBasisFunction(nn.Module):
    def __init__(
        self,
        grid_min: float = -2.0,
        grid_max: float = 2.0,
        num_grids: int = 4,
        denominator: float = None,
    ):
        super().__init__()
        grid = torch.linspace(grid_min, grid_max, num_grids)
        self.grid = torch.nn.Parameter(grid, requires_grad=False)
        self.denominator = denominator or (grid_max - grid_min) / (num_grids - 1)

    def forward(self, x):
        return torch.exp(-((x[..., None] - self.grid) / self.denominator) ** 2)


class SplineConv2D(nn.Conv2d):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: Union[int, Tuple[int, int]] = 3,
        stride: Union[int, Tuple[int, int]] = 1,
        padding: Union[int, Tuple[int, int]] = 0,
        dilation: Union[int, Tuple[int, int]] = 1,
        groups: int = 1,
        bias: bool = True,
        init_scale: float = 0.1,
        padding_mode: str = "zeros",
        **kw
    ) -> None:
        self.init_scale = init_scale
        super().__init__(
            in_channels,
            out_channels,
            kernel_size,
            stride,
            padding,
            dilation,
            groups,
            bias,
            padding_mode,
            **kw
        )

    def reset_parameters(self) -> None:
        nn.init.trunc_normal_(self.weight, mean=0, std=self.init_scale)
        if self.bias is not None:
            nn.init.zeros_(self.bias)


class FastKANConvLayer(nn.Module):
    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: Union[int, Tuple[int, int]] = 3,
        stride: Union[int, Tuple[int, int]] = 1,
        padding: Union[int, Tuple[int, int]] = 0,
        dilation: Union[int, Tuple[int, int]] = 1,
        groups: int = 1,
        bias: bool = True,
        grid_min: float = -2.0,
        grid_max: float = 2.0,
        num_grids: int = 4,
        use_base_update: bool = True,
        base_activation=F.silu,
        spline_weight_init_scale: float = 0.1,
        padding_mode: str = "zeros",
        kan_type: str = "RBF",
    ) -> None:
        super().__init__()
        if kan_type == "RBF":
            self.rbf = RadialBasisFunction(grid_min, grid_max, num_grids)
        elif kan_type == "Fourier":
            self.rbf = FourierBasisFunction(num_grids)
        elif kan_type == "Poly":
            self.rbf = PolynomialFunction(num_grids)
        elif kan_type == "Chebyshev":
            self.rbf = ChebyshevFunction(num_grids)
        elif kan_type == "BSpline":
            self.rbf = BSplineFunction(grid_min, grid_max, 3, num_grids)

        self.spline_conv = SplineConv2D(
            in_channels * num_grids,
            out_channels,
            kernel_size,
            stride,
            padding,
            dilation,
            groups,
            bias,
            spline_weight_init_scale,
            padding_mode,
        )

        self.use_base_update = use_base_update
        if use_base_update:
            self.base_activation = base_activation
            self.base_conv = nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size,
                stride,
                padding,
                dilation,
                groups,
                bias,
                padding_mode,
            )

    def forward(self, x):
        batch_size, channels, height, width = x.shape
        x_rbf = self.rbf(x.view(batch_size, channels, -1)).view(
            batch_size, channels, height, width, -1
        )
        x_rbf = x_rbf.permute(0, 4, 1, 2, 3).contiguous().view(
            batch_size, -1, height, width
        )

        # Apply spline convolution
        ret = self.spline_conv(x_rbf)

        if self.use_base_update:
            base = self.base_conv(self.base_activation(x))
            ret = ret + base

        return ret


"""
Pytorch implementation of U-Net based on Kolmogorov-Arnold Network, based on
the U-Net implementation in https://github.com/milesial/Pytorch-UNet.
The U-Net model is modified to use the FastKANConvLayer instead of the Conv2d layer in
the original implementation.
The Convolution operation is implemented in https://github.com/XiangboGaoBarry/KA-Conv
"""


class DoubleConv(nn.Module):
    """(convolution => [BN] => ReLU) * 2"""

    def __init__(self, in_channels, out_channels, device):
        super().__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.device = device

        self.double_conv = nn.Sequential(
            FastKANConvLayer(self.in_channels, self.out_channels // 2, padding=1, kernel_size=3, stride=1, kan_type="RBF"),
            nn.BatchNorm2d(self.out_channels // 2),
            nn.ReLU(inplace=True),
            FastKANConvLayer(self.out_channels // 2, self.out_channels, padding=1, kernel_size=3, stride=1, kan_type="RBF"),
            nn.BatchNorm2d(self.out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.double_conv(x)


class Down(nn.Module):
    """Downscaling with maxpool then double conv"""

    def __init__(self, in_channels, out_channels, device="mps"):
        super().__init__()

        self.device = device
        self.maxpool_conv = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_channels, out_channels, device=self.device),
        )

    def forward(self, x):
        return self.maxpool_conv(x)


class Up(nn.Module):
    """Upscaling then double conv"""

    def __init__(self, in_channels, out_channels, bilinear=True, device="mps"):
        super().__init__()

        # if bilinear, use the normal convolutions to reduce the number of channels
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
            self.conv = DoubleConv(in_channels, out_channels, device=device)
        else:
            self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        # input is CHW
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]

        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2, diffY // 2, diffY - diffY // 2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class OutConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(OutConv, self).__init__()
        self.conv = FastKANConvLayer(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)


class KANU_Net(nn.Module):
    def __init__(self, n_channels, n_classes, bilinear=True, device="mps"):
        super(KANU_Net, self).__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear
        self.device = device

        self.channels = [64, 128, 256, 512, 1024]

        self.inc = DoubleConv(n_channels, 64, device=self.device)

        self.down1 = Down(self.channels[0], self.channels[1], self.device)
        self.down2 = Down(self.channels[1], self.channels[2], self.device)
        self.down3 = Down(self.channels[2], self.channels[3], self.device)
        factor = 2 if bilinear else 1
        self.down4 = Down(self.channels[3], self.channels[4] // factor, self.device)
        self.up1 = Up(self.channels[4], self.channels[3] // factor, bilinear, self.device)
        self.up2 = Up(self.channels[3], self.channels[2] // factor, bilinear, self.device)
        self.up3 = Up(self.channels[2], self.channels[1] // factor, bilinear, self.device)
        self.up4 = Up(self.channels[1], self.channels[0], bilinear, self.device)
        self.outc = OutConv(self.channels[0], n_classes)

    def forward(self, x):
        # Encoder
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        #Decoder
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        logits = self.outc(x)
        return logits


## Losses, Optimizer, and Metrics

In [5]:
class DiceLoss(nn.Module):
    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, logits, targets):
        num_classes = logits.shape[1]
        probs = torch.softmax(logits, dim=1)
        targets_one_hot = F.one_hot(targets, num_classes).permute(0, 3, 1, 2).float()
        dims = (0, 2, 3)
        intersection = torch.sum(probs * targets_one_hot, dims)
        cardinality = torch.sum(probs + targets_one_hot, dims)
        dice = (2.0 * intersection + self.smooth) / (cardinality + self.smooth)
        return 1.0 - dice.mean()


class SegmentationMetrics:
    def __init__(self, num_classes, ignore_index=None, eps=1e-7):
        self.num_classes = num_classes
        self.ignore_index = ignore_index
        self.eps = eps
        self.reset()

    def reset(self):
        self.tp = torch.zeros(self.num_classes, dtype=torch.float64)
        self.fp = torch.zeros(self.num_classes, dtype=torch.float64)
        self.fn = torch.zeros(self.num_classes, dtype=torch.float64)
        self.correct = 0
        self.total = 0

    def update(self, logits, targets):
        preds = torch.argmax(logits, dim=1)
        if self.ignore_index is not None:
            valid_mask = targets != self.ignore_index
            preds = preds[valid_mask]
            targets = targets[valid_mask]
        self.correct += (preds == targets).sum().item()
        self.total += targets.numel()
        for cls in range(self.num_classes):
            pred_cls = preds == cls
            target_cls = targets == cls
            self.tp[cls] += (pred_cls & target_cls).sum().item()
            self.fp[cls] += (pred_cls & ~target_cls).sum().item()
            self.fn[cls] += (~pred_cls & target_cls).sum().item()

    def compute(self):
        precision = self.tp / (self.tp + self.fp + self.eps)
        recall = self.tp / (self.tp + self.fn + self.eps)
        f1 = 2 * precision * recall / (precision + recall + self.eps)
        iou = self.tp / (self.tp + self.fp + self.fn + self.eps)
        dice = 2 * self.tp / (2 * self.tp + self.fp + self.fn + self.eps)
        accuracy = self.correct / max(self.total, 1)
        return {
            "accuracy": float(accuracy),
            "precision": float(precision.mean()),
            "recall": float(recall.mean()),
            "f1": float(f1.mean()),
            "iou": float(iou.mean()),
            "dice": float(dice.mean()),
            "dice_loss": float(1.0 - dice.mean()),
        }


model = KANU_Net(n_channels=3, n_classes=NUM_CLASSES, bilinear=True, device=DEVICE.type).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=5, factor=0.5)

criterion_ce = nn.CrossEntropyLoss()
criterion_dice = DiceLoss()

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Model parameters: 33,471,024


## DataLoaders

In [6]:
num_workers = min(4, os.cpu_count() or 1)

train_dataset = MassachusettsBuildingsDataset(
    TRAIN_IMG_DIR,
    TRAIN_MASK_DIR,
    img_size=IMG_SIZE,
    variant=DATASET_VARIANT,
)
val_dataset = MassachusettsBuildingsDataset(
    VAL_IMG_DIR,
    VAL_MASK_DIR,
    img_size=IMG_SIZE,
    variant=DATASET_VARIANT,
)
test_dataset = MassachusettsBuildingsDataset(
    TEST_IMG_DIR,
    TEST_MASK_DIR,
    img_size=IMG_SIZE,
    variant=DATASET_VARIANT,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=torch.cuda.is_available(),
)

print(f"Train samples: {len(train_dataset)}")
print(f"Val samples: {len(val_dataset)}")
print(f"Test samples: {len(test_dataset)}")

Train samples: 137
Val samples: 4
Test samples: 10


In [7]:
def format_metrics(prefix, metrics):
    return (
        f"{prefix} acc={metrics['accuracy']:.4f} "
        f"prec={metrics['precision']:.4f} recall={metrics['recall']:.4f} "
        f"f1={metrics['f1']:.4f} dice={metrics['dice']:.4f} "
        f"iou={metrics['iou']:.4f} dice_loss={metrics['dice_loss']:.4f}"
    )


def run_epoch(loader, model, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    metrics = SegmentationMetrics(NUM_CLASSES, ignore_index=IGNORE_INDEX)
    total_loss = 0.0
    total_steps = len(loader)

    if is_train:
        optimizer.zero_grad(set_to_none=True)

    with torch.set_grad_enabled(is_train):
        for step, (images, masks) in enumerate(loader, start=1):
            images = images.to(DEVICE, non_blocking=True)
            masks = masks.to(DEVICE, non_blocking=True)

            with torch.cuda.amp.autocast(enabled=USE_AMP):
                logits = model(images)
                loss_ce = criterion_ce(logits, masks)
                loss_dice = criterion_dice(logits, masks)
                loss = loss_ce + loss_dice

            if is_train:
                scaled_loss = loss / ACCUM_STEPS
                scaler.scale(scaled_loss).backward()
                if step % ACCUM_STEPS == 0 or step == total_steps:
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)

            total_loss += loss.item() * images.size(0)
            metrics.update(logits, masks)

    avg_loss = total_loss / max(len(loader.dataset), 1)
    return avg_loss, metrics.compute()

## Training Loop

In [8]:
history = {"train_loss": [], "val_loss": [], "train_metrics": [], "val_metrics": []}
best_val_loss = float("inf")
best_model_path = os.path.join(MODEL_SAVE_DIR, "kan-unet-best.pth")
train_metrics_path = os.path.join(MODEL_SAVE_DIR, "metrics_train_val.csv")
best_train_loss = float("inf")
best_train_epoch = 0

train_start = time.time()
print("Training start:", time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(train_start)))

with open(train_metrics_path, "w", newline="") as metrics_file:
    writer = csv.writer(metrics_file)
    writer.writerow(["epoch", "split", "loss", "accuracy", "precision", "recall", "f1", "dice", "iou", "dice_loss"])

    for epoch in range(EPOCHS):
        epoch_start = time.time()
        train_loss, train_metrics = run_epoch(train_loader, model, optimizer=optimizer)
        val_loss, val_metrics = run_epoch(val_loader, model, optimizer=None)
        scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_metrics"].append(train_metrics)
        history["val_metrics"].append(val_metrics)

        if train_loss < best_train_loss:
            best_train_loss = train_loss
            best_train_epoch = epoch + 1

        writer.writerow([
            epoch + 1,
            "train",
            train_loss,
            train_metrics["accuracy"],
            train_metrics["precision"],
            train_metrics["recall"],
            train_metrics["f1"],
            train_metrics["dice"],
            train_metrics["iou"],
            train_metrics["dice_loss"],
        ])
        writer.writerow([
            epoch + 1,
            "val",
            val_loss,
            val_metrics["accuracy"],
            val_metrics["precision"],
            val_metrics["recall"],
            val_metrics["f1"],
            val_metrics["dice"],
            val_metrics["iou"],
            val_metrics["dice_loss"],
        ])
        metrics_file.flush()

        print(f"Epoch {epoch + 1}/{EPOCHS} | train_loss={train_loss:.4f} val_loss={val_loss:.4f}")
        print(format_metrics("Train", train_metrics))
        print(format_metrics("Val", val_metrics))
        print(f"Epoch time: {time.time() - epoch_start:.1f}s")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(
                {
                    "epoch": epoch + 1,
                    "model_state_dict": model.state_dict(),
                    "optimizer_state_dict": optimizer.state_dict(),
                    "val_loss": best_val_loss,
                },
                best_model_path,
            )
            print(f"Saved best model: {best_model_path}")

train_end = time.time()
print("Training end:", time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(train_end)))
print(f"Total training time: {train_end - train_start:.1f}s")
print(f"Best train epoch: {best_train_epoch} (train_loss={best_train_loss:.4f})")
print(f"Train/val metrics saved to {train_metrics_path}")

Training start: 2026-06-06 11:13:29


/tmp/ipykernel_146670/4192161215.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


Epoch 1/50 | train_loss=1.5161 val_loss=1.2022
Train acc=0.3355 prec=0.5322 recall=0.5519 f1=0.3270 dice=0.3270 iou=0.1979 dice_loss=0.6730
Val acc=0.6154 prec=0.5568 recall=0.6432 f1=0.5076 dice=0.5076 iou=0.3729 dice_loss=0.4924
Epoch time: 15.4s
Saved best model: /media/aejaz/New Volume/Projects/Massachusetts/KAN-UNET/models/kan-unet-best.pth
Epoch 2/50 | train_loss=1.1727 val_loss=0.9388
Train acc=0.6442 prec=0.5563 recall=0.6153 f1=0.5305 dice=0.5305 iou=0.3955 dice_loss=0.4695
Val acc=0.8588 prec=0.6162 recall=0.5987 f1=0.6062 dice=0.6062 iou=0.5124 dice_loss=0.3938
Epoch time: 14.5s
Saved best model: /media/aejaz/New Volume/Projects/Massachusetts/KAN-UNET/models/kan-unet-best.pth
Epoch 3/50 | train_loss=0.9682 val_loss=0.8248
Train acc=0.8205 prec=0.5995 recall=0.5937 f1=0.5964 dice=0.5964 iou=0.4935 dice_loss=0.4036
Val acc=0.8943 prec=0.7299 recall=0.5614 f1=0.5810 dice=0.5810 iou=0.5077 dice_loss=0.4190
Epoch time: 14.5s
Saved best model: /media/aejaz/New Volume/Projects/Mass

## Test Evaluation

In [9]:
if os.path.isfile(best_model_path):
    checkpoint = torch.load(best_model_path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state_dict"])
    print(f"Loaded best model from {best_model_path}")

test_loss, test_metrics = run_epoch(test_loader, model, optimizer=None)
print(f"Test loss: {test_loss:.4f}")
print(format_metrics("Test", test_metrics))

test_metrics_path = os.path.join(MODEL_SAVE_DIR, "metrics_test.csv")
with open(test_metrics_path, "w", newline="") as metrics_file:
    writer = csv.writer(metrics_file)
    writer.writerow(["split", "loss", "accuracy", "precision", "recall", "f1", "dice", "iou", "dice_loss"])
    writer.writerow([
        "test",
        test_loss,
        test_metrics["accuracy"],
        test_metrics["precision"],
        test_metrics["recall"],
        test_metrics["f1"],
        test_metrics["dice"],
        test_metrics["iou"],
        test_metrics["dice_loss"],
    ])
print(f"Test metrics saved to {test_metrics_path}")

Loaded best model from /media/aejaz/New Volume/Projects/Massachusetts/KAN-UNET/models/kan-unet-best.pth


/tmp/ipykernel_146670/4192161215.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=USE_AMP):


Test loss: 0.7376
Test acc=0.8453 prec=0.7560 recall=0.6685 f1=0.6957 dice=0.6957 iou=0.5755 dice_loss=0.3043
Test metrics saved to /media/aejaz/New Volume/Projects/Massachusetts/KAN-UNET/models/metrics_test.csv


## Save Final Model

In [10]:
final_model_path = os.path.join(MODEL_SAVE_DIR, "kan-unet-final.pth")
torch.save(model.state_dict(), final_model_path)
print(f"Final model saved to {final_model_path}")

Final model saved to /media/aejaz/New Volume/Projects/Massachusetts/KAN-UNET/models/kan-unet-final.pth


In [8]:

from pathlib import Path
from PIL import Image, ImageDraw
import numpy as np
import torch

EXPORT_DPI  = 600
EXPORT_SIZE = 2048
OUTPUT_DIR  = Path(MODEL_SAVE_DIR).parent / "comparison_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Load best weights ─────────────────────────────────────────
best_model_path = os.path.join(MODEL_SAVE_DIR, "kan-unet-best.pth")
if Path(best_model_path).is_file():
    checkpoint = torch.load(best_model_path, map_location=DEVICE)
    state_dict = checkpoint.get("model_state_dict", checkpoint)
    model.load_state_dict(state_dict)
    print(f"Loaded best model from {best_model_path}")
else:
    print("[WARN] best model not found — using current weights")

# ── Inference ─────────────────────────────────────────────────
# Dataset: image loaded as RGB → ToTensor() → 3 x H x W float32
# Model:   KANU_Net(n_channels=3, n_classes=2) → B x 2 x H x W logits
# Loss:    CrossEntropyLoss → decode with argmax(dim=1)
model.eval()
image_tensor, mask_tensor = test_dataset[0]       # 3xHxW float32, HxW int64
image_batch = image_tensor.unsqueeze(0).to(DEVICE)

with torch.no_grad():
    logits = model(image_batch)                   # 1 x 2 x H x W

if isinstance(logits, (tuple, list)):
    logits = logits[0]

# 2-class output → argmax → class index 0 (bg) or 1 (building)
pred_mask = torch.argmax(logits, dim=1)[0]        # H x W, values 0/1

# ── Helpers ───────────────────────────────────────────────────
def _to_rgb_image(tensor):
    """3 x H x W [0,1] tensor → HWC uint8. No de-norm (ToTensor only in training)."""
    arr = tensor.detach().cpu().clamp(0, 1).permute(1, 2, 0).numpy()
    return (arr * 255).round().astype(np.uint8)

def _to_bw_mask(tensor):
    """Int64 class-index tensor or bool → 0/255 uint8."""
    array = tensor.detach().cpu().squeeze().numpy()
    binary = array > 0.5 if array.max() <= 1 else array > 0
    return (binary.astype(np.uint8) * 255)

def _save_png_600dpi(array, path, resample):
    image = Image.fromarray(array)
    if image.size != (EXPORT_SIZE, EXPORT_SIZE):
        image = image.resize((EXPORT_SIZE, EXPORT_SIZE), resample=resample)
    image.save(path, dpi=(EXPORT_DPI, EXPORT_DPI))
    return image

def _with_title(image, title):
    rgb          = image.convert("RGB")
    title_height = 128
    canvas       = Image.new("RGB", (rgb.width, rgb.height + title_height), "white")
    canvas.paste(rgb, (0, title_height))
    draw = ImageDraw.Draw(canvas)
    draw.rectangle((0, 0, rgb.width, title_height), fill="black")
    draw.text((40, 42), title, fill="white")
    return canvas

# ── Build arrays ──────────────────────────────────────────────
stem = "test_sample"
if hasattr(test_dataset, "pairs") and test_dataset.pairs:
    stem = Path(test_dataset.pairs[0][0]).stem

input_rgb       = _to_rgb_image(image_tensor)
ground_truth_bw = _to_bw_mask(mask_tensor)
prediction_bw   = _to_bw_mask(pred_mask)

# ── Save individual PNGs ──────────────────────────────────────
input_image        = _save_png_600dpi(input_rgb,       OUTPUT_DIR / f"{stem}_input_600dpi.png",        Image.Resampling.BICUBIC)
ground_truth_image = _save_png_600dpi(ground_truth_bw, OUTPUT_DIR / f"{stem}_ground_truth_600dpi.png", Image.Resampling.NEAREST)
prediction_image   = _save_png_600dpi(prediction_bw,   OUTPUT_DIR / f"{stem}_prediction_600dpi.png",   Image.Resampling.NEAREST)

# ── Side-by-side comparison ───────────────────────────────────
panels = [
    _with_title(input_image,        "Input"),
    _with_title(ground_truth_image, "Ground Truth"),
    _with_title(prediction_image,   "Prediction"),
]
comparison = Image.new(
    "RGB",
    (sum(p.width for p in panels), max(p.height for p in panels)),
    "white",
)
x = 0
for panel in panels:
    comparison.paste(panel, (x, 0))
    x += panel.width

comparison_path = OUTPUT_DIR / f"{stem}_comparison_600dpi.png"
comparison.save(comparison_path, dpi=(EXPORT_DPI, EXPORT_DPI))

print(f"Saved input:        {OUTPUT_DIR / (stem + '_input_600dpi.png')}")
print(f"Saved ground truth: {OUTPUT_DIR / (stem + '_ground_truth_600dpi.png')}")
print(f"Saved prediction:   {OUTPUT_DIR / (stem + '_prediction_600dpi.png')}")
print(f"Saved comparison:   {comparison_path}")

Loaded best model from /media/aejaz/New Volume/Projects/Massachusetts/KAN-UNET/models/kan-unet-best.pth
Saved input:        /media/aejaz/New Volume/Projects/Massachusetts/KAN-UNET/comparison_outputs/22828930_15_input_600dpi.png
Saved ground truth: /media/aejaz/New Volume/Projects/Massachusetts/KAN-UNET/comparison_outputs/22828930_15_ground_truth_600dpi.png
Saved prediction:   /media/aejaz/New Volume/Projects/Massachusetts/KAN-UNET/comparison_outputs/22828930_15_prediction_600dpi.png
Saved comparison:   /media/aejaz/New Volume/Projects/Massachusetts/KAN-UNET/comparison_outputs/22828930_15_comparison_600dpi.png
